# 🇪🇺 Aᵢ Component in EU General Allocation  
### *Social, Economic and Territorial Cohesion*

This formula defines the **Aᵢ component**, used to allocate general EU funds among Member States  
based on **population**, **rural poverty**, **income levels**, and **regional disparities**.

---

## 🔷 Formula

$$
A_i = \text{average} \left( \frac{\text{Pop}_i}{\text{Pop}_{EU}}, \frac{\text{AROPE\_ra}_i}{\text{AROPE\_ra}_{EU}} \right)
\times
\left[
\frac{\text{GNI}_{pc\ PPS\ EU}}{\text{GNI}_{pc\ PPS\ i}} \cdot \left(1 + \text{Regional prosperity gap}_i + \text{Agri prosperity gap}_i \right)
\right]^2
$$

---

## 🟦 Regional Prosperity Gap

$$
\text{Regional prosperity gap}_i =
\frac{1}{\text{Pop}_i} \sum_r \left( \max \left(0,\ 0.75 - \frac{\text{GDP}_{pc\ PPS\ r}}{\text{GDP}_{pc\ PPS\ EU}} \right) \cdot \text{Pop}_r \right)
$$

- Assesses underperforming regions (GDP/pc < 75% of EU average)  
- Weighted by regional population within Member State *i*

---

## 🟩 Agri Prosperity Gap

$$
\text{Agri prosperity gap}_i =
\max \left(0,\ 0.90 \cdot \frac{DP}{ha}_{EU} - \frac{DP}{ha}_i \right) \cdot \frac{ha_i}{DP_i}
$$

- Compares national average direct payment per hectare to 90% of EU average  
- Scaled by eligible area and total direct payments

---

## 📘 Definitions & Parameters

- **Pop<sub>i</sub>**: National population (Eurostat `demo_gind`)
- **AROPE<sub>ra</sub><sub>i</sub>**: Population at risk of poverty or social exclusion in rural areas (Eurostat `ilc_peps13n`)
- **GNI pc PPS**: Gross national income per capita, purchasing power standard (Eurostat `nama_10_pp`)
- **GDP pc PPS<sub>r</sub>**: Regional GDP per capita (Eurostat `nama_10r_3gdp`, 2021–2023 avg)
- **DP<sub>i</sub>**: Direct payments (2027), excluding POSEI/SAI
- **DP/ha<sub>i</sub>**: Direct payment per eligible hectare in Member State *i*
- **ha<sub>i</sub>**: Declared eligible hectares (claim year 2022)

---

## ⚖️ Normalisation & Caps

- All **Aᵢ values are normalised** so that:
  $$
  \sum_i \alpha_i = 100\%
  $$

- **Capping rules:**
  - A Member State’s allocation share **cannot be less than 80%** or **more than 105%** of its **2021–2027 share**
  - If exceeded, **proportional adjustment** is applied

---

## 📊 Summary Table

| Component                  | Description                                                  |
|---------------------------|--------------------------------------------------------------|
| Population share          | 2024 population relative to EU total                         |
| Rural poverty share       | AROPE in rural areas vs EU average                           |
| Income adjustment         | Inverse of GNI/pc in PPS                                     |
| Regional prosperity gap   | Weighted GDP/pc shortfall in NUTS3 regions                   |
| Agri prosperity gap       | Deficit in DP/ha compared to 90% of EU average               |

In [ ]:
from mff.general_allocation import (
    calculate_agri_prosperity_gap,
    calculate_general_allocation,
    calculate_gni_multiplier,
    calculate_product_part1,
    calculate_regional_prosperity_gap,
    gradient_row,
    read_data_country_lvl,
    read_data_nuts3_lvl,
)

In [ ]:
database_country = read_data_country_lvl()
database_nuts3 = read_data_nuts3_lvl()

agri_prosperity_gap = calculate_agri_prosperity_gap(database_country)
regional_prosperity_gap = calculate_regional_prosperity_gap(database_nuts3)
gni_multiplier = calculate_gni_multiplier(database_country)
product_part1 = calculate_product_part1(database_country)

general_allocation = (
    calculate_general_allocation(
        agri_prosperity_gap, regional_prosperity_gap, gni_multiplier, product_part1
    )
    .to_pandas()
    .sort_values(by="geo_codes")
)

general_allocation.style.apply(gradient_row, axis=1)